# rehydrate compaction moment

This notebook runs one concrete, owner-authorized compaction check from a local private Codex rollout. Recovery and judging are wired directly to the datapoint below.

Target datapoint:

- snapshot: `/Users/gabriel/.codex/sessions/2026/05/20/rollout-2026-05-20T16-01-40-019e45b1-2dc0-7ea3-b7b5-694ac2f586e9.jsonl`
- compaction line: `912`

That private rollout is intentionally not committed. The committed `evidence/current-codex-chat-rollout.jsonl` file is only a redacted structural sample.


In [8]:
from __future__ import annotations

import importlib
from pathlib import Path

import pandas as pd
from IPython.display import display

import baseline_helpers as bh

bh = importlib.reload(bh)

ROOT = Path.cwd()
SNAPSHOT = Path.home() / ".codex/sessions/2026/05/20/rollout-2026-05-20T16-01-40-019e45b1-2dc0-7ea3-b7b5-694ac2f586e9.jsonl"
TARGET_COMPACTION_LINE = 912
JUDGE_MODEL = bh.default_judge_model()

if not SNAPSHOT.exists():
    raise FileNotFoundError(
        f"Private authorized rollout not found: {SNAPSHOT}. "
        "Do not commit this file; keep it local or under private/."
    )
if not bh.has_openai_api_key():
    raise RuntimeError("OPENAI_API_KEY is required in .env or the environment")

pd.set_option("display.max_colwidth", 180)

pd.DataFrame([
    {
        "snapshot": str(SNAPSHOT),
        "target_compaction_line": TARGET_COMPACTION_LINE,
        "judge_model": JUDGE_MODEL,
    }
])

,snapshot,target_compaction_line,judge_model
0,/Users/gabriel/.codex/sessions/2026/05/20/rollout-2026-05-20T16-01-40-019e45b1-2dc0-7ea3-b7b5-694ac2f586e9.jsonl,912,gpt-5.5


## Dataset Profile

The notebook profiles the private rollout directly from raw JSONL lines.

In [9]:
profile = bh.profile_rollout(SNAPSHOT)
rows = pd.DataFrame(profile["rows"])
summary = pd.DataFrame(profile["summary_rows"])
events = pd.DataFrame(profile["event_type_rows"])
payloads = pd.DataFrame(profile["payload_type_rows"])
schemas = pd.DataFrame(profile["schema_rows"])
compactions = pd.DataFrame(profile["compaction_rows"])

summary

,metric,value
0,source file,rollout-2026-05-20T16-01-40-019e45b1-2dc0-7ea3-b7b5-694ac2f586e9.jsonl
1,snapshot sha256,b1377cc6f7499c2b71c23f968925c9ba6db434325258d31c72771c8d3a5d6477
2,source lines,1396
3,valid JSON object lines,1396
4,invalid JSON lines,0
5,event types observed,5
6,payload types observed,15
7,turn_context events,13
8,compaction-related events,2
9,discoverable session ids,019e45b1-2dc0-7ea3-b7b5-694ac2f586e9


## Event Surface

In [10]:
event_surface = (
    events.head(8)
    .rename(columns={"value": "event_type"})
    .assign(kind="event_type")[["kind", "event_type", "count", "percent"]]
)
payload_surface = (
    payloads.head(8)
    .rename(columns={"value": "event_type"})
    .assign(kind="payload_type")[["kind", "event_type", "count", "percent"]]
)
surface = pd.concat([event_surface, payload_surface], ignore_index=True)

schema_shape = pd.DataFrame([
    {
        "observed_event_types": len(events),
        "observed_payload_types": len(payloads),
        "observed_event_schema_shapes": len(schemas),
        "first_compaction_line": int(compactions["line_number"].min()) if not compactions.empty else None,
    }
])

display(schema_shape)
surface

,observed_event_types,observed_payload_types,observed_event_schema_shapes,first_compaction_line
0,5,15,5,912


,kind,event_type,count,percent
0,event_type,response_item,983,70.42
1,event_type,event_msg,398,28.51
2,event_type,turn_context,13,0.93
3,event_type,compacted,1,0.07
4,event_type,session_meta,1,0.07
5,payload_type,function_call,316,22.64
6,payload_type,function_call_output,316,22.64
7,payload_type,token_count,234,16.76
8,payload_type,reasoning,153,10.96
9,payload_type,message,92,6.59


## Selected Compaction

This is the source history plus the full encrypted compaction artifact for the datapoint being judged.

In [11]:
survival_cases = bh.compaction_survival_cases(profile)
server_recovery_cases = bh.compaction_server_recovery_cases(profile)

selected_cases = [
    case for case in survival_cases
    if case["compaction_line"] == TARGET_COMPACTION_LINE
]
if not selected_cases:
    raise RuntimeError(f"No survival case found at line {TARGET_COMPACTION_LINE}")
selected_case = selected_cases[0]

selected_recovery_cases = [
    case for case in server_recovery_cases
    if case["compaction_line"] == TARGET_COMPACTION_LINE
    and case["encrypted_content_available"]
]
if not selected_recovery_cases:
    raise RuntimeError(f"No full encrypted_content artifact found at line {TARGET_COMPACTION_LINE}")

compaction_inputs = pd.DataFrame(bh.compaction_survival_case_rows([selected_case]))
server_recovery_inputs = pd.DataFrame(
    bh.compaction_server_recovery_case_rows(selected_recovery_cases)
)
artifact_summary = pd.DataFrame([
    {
        "compaction_line": TARGET_COMPACTION_LINE,
        "replacement_history_items": len(selected_case["source"]["items"]),
        "plaintext_message_before_recovery": selected_case["compacted_artifact"]["plaintext_message_available"],
        "full_server_artifacts_available": len(selected_recovery_cases),
        "encrypted_content_chars": selected_recovery_cases[-1]["encrypted_content_chars"],
    }
])

display(artifact_summary)
display(server_recovery_inputs)
compaction_inputs

,compaction_line,replacement_history_items,plaintext_message_before_recovery,full_server_artifacts_available,encrypted_content_chars
0,912,13,False,1,18680


,case_id,compaction_line,replacement_history_index,encrypted_content_chars,encrypted_content_available,source_ref
0,server-compaction-recovery:line-912,912,12,18680,True,line-912:payload.replacement_history[12]


,case_id,compaction_line,replacement_history_items,plaintext_message_available,plaintext_message_chars,opaque_compaction_items,opaque_compaction_chars,auditable_survival_target
0,compaction-survival:line-912,912,13,False,0,1,18680,False


## Recover Compacted Message

The Responses API accepts the selected `type="compaction"` item and returns a user-visible compacted message for judging.

In [12]:
server_recovery_result = bh.recover_last_compacted_message(
    {
        **profile,
        "rows": [
            row for row in profile["rows"]
            if row["line_number"] == TARGET_COMPACTION_LINE
        ],
    },
    model=JUDGE_MODEL,
)

# The filtered profile above forces recovery of this exact selected datapoint.
if server_recovery_result.get("status") != "ok":
    raise RuntimeError(server_recovery_result.get("error"))

recovered = server_recovery_result.get("result") or {}
if recovered.get("recovered") is not True:
    raise RuntimeError("Artifact was accepted but no compacted message was recovered")

recovered_survival_case = bh.compaction_case_with_recovered_message(
    selected_case,
    server_recovery_result,
)

pd.DataFrame(bh.server_recovery_status_rows(server_recovery_result))

,status,recovered,confidence,response_id,model,candidate_source_kind,candidate_source,candidate_compaction_line,candidate_replacement_history_index,artifact_chars,artifact_sha256_12,openai_call_count,input_tokens,output_tokens,total_tokens,error
0,ok,True,high,resp_0c728fa256bc12ee016a0de01a050c81a094f70214183066e8,gpt-5.5,snapshot,line-912:payload.replacement_history[12],912,12,18680,5391f334f1f1,1,3123,2125,5248,None


## Judge Compaction Quality

The judge extracts important explicit facts from `payload.replacement_history` and scores whether those facts survived in the recovered compacted message.

In [13]:
survival_response = bh.run_openai_compaction_survival_judges(
    [recovered_survival_case],
    model=JUDGE_MODEL,
)

if survival_response["status"] != "ok":
    failed = [
        row for row in bh.compaction_survival_call_rows(survival_response)
        if row["status"] != "ok"
    ]
    display(pd.DataFrame(failed))
    raise RuntimeError(f"survival judge failed for {len(failed)} compaction case(s)")

if survival_response.get("openai_call_count") != 1:
    raise RuntimeError("this notebook must use exactly one judge call")

survival_result = bh.combine_compaction_survival_results(survival_response)

display(pd.DataFrame(bh.compaction_survival_call_rows(survival_response)))
display(pd.DataFrame(bh.compaction_survival_summary_rows(survival_result)))
pd.DataFrame(bh.compaction_survival_result_rows(survival_result))

,case_id,status,response_id,model,error,input_tokens,output_tokens,total_tokens
0,compaction-survival:line-912:server-recovered,ok,resp_08316ce618175f75016a0de033c1dc8195ad2dbacc0989c0fd,gpt-5.5,None,21140,1976,23116


,case_id,compaction_line,compacted_artifact_available,facts,preserved,partial,lost,survival_score_0_to_2,rationale
0,compaction-survival:line-912:server-recovered,912,True,6,5,1,0,2,"The compacted artifact preserves the main actionable context from the readable replacement history: branch, PR goal, install-skills formats, auto-registered MCP skill resources..."


,case_id,compaction_line,survival_score_0_to_2,fact_id,survival,fact,source_refs,raw_line_ids,evidence
0,compaction-survival:line-912:server-recovered,912,2,F1,preserved,"The approved PR plan was to add `taurus-cli install-skills`, move workflow/skill markdown into a shared `internal/skills/` package, and have the MCP server auto-register embedd...",line-912:payload.replacement_history[1],912,"The artifact says current PR contents include a new `internal/skills/` package, `taurus-cli install-skills --format claude|cursor|codex`, and that the MCP server auto-registers..."
1,compaction-survival:line-912:server-recovered,912,2,F2,partial,"The work was to happen on a new branch forked from `add-governance-tools`, and a branch named `install-skills` was created.","line-912:payload.replacement_history[0], line-912:payload.replacement_history[1]",912,"The artifact preserves the active branch as `install-skills`, but does not preserve that it was forked from `add-governance-tools`."
2,compaction-survival:line-912:server-recovered,912,2,F3,preserved,"The install command was planned to support Claude Code, Cursor, and Codex via `--format claude|cursor|codex`, generating target-appropriate skill folders/files for each platform.","line-912:payload.replacement_history[1], line-912:payload.replacement_history[2], line-912:payload.replacement_history[3]",912,The artifact explicitly lists `taurus-cli install-skills --format claude|cursor|codex` and describes generated skills as part of the PR.
3,compaction-survival:line-912:server-recovered,912,2,F4,preserved,"The user wanted templating so CLI and MCP tool names would be exactly right, and questioned whether the implementation was using the fact that the tools are shared/tied together.","line-912:payload.replacement_history[5], line-912:payload.replacement_history[6]",912,"The artifact says templating support exists in `internal/skills/skills.go` with `{{tool ...}}`, `{{call ...}}`, `{{param ...}}`, backed by `internal/tools`, plus `tools.CLIName..."
4,compaction-survival:line-912:server-recovered,912,2,F5,preserved,"The user challenged the statement that real CLI flags for nested JSON inputs would require broader CLI argument support, asking why it could not be done.",line-912:payload.replacement_history[8],912,"The artifact explicitly says the user challenged the claim that exact CLI flags for nested JSON inputs were not doable, and records the decision to treat complex tool fields as..."
5,compaction-survival:line-912:server-recovered,912,2,F6,preserved,The user’s latest direct instruction was to polish the PR and make it clean/neat.,"line-912:payload.replacement_history[4], line-912:payload.replacement_history[7], line-912:payload.replacement_history[11]",912,The artifact says the user wants the PR polished/clean and dislikes hand-wavy updates.
